# T5 i inne rodziny transformerów

## Zadanie
Poznaj encoder–decoder T5 i porównaj go z GPT. Zobacz przepływ encoder self-attention, decoder causal attention, cross-attention oraz cel span corruption. Kod PyTorcha demonstruje tensorowe interfejsy, nie odtwarza pełnego T5.

## 1. Encoder-only, decoder-only, encoder–decoder
- **BERT / encoder-only:** dwukierunkowa reprezentacja wejścia; klasyfikacja i ekstrakcja.
- **GPT / decoder-only:** causal self-attention; kontynuacja i generowanie.
- **T5 / encoder–decoder:** encoder czyta źródło, decoder generuje cel i pobiera kontekst przez cross-attention.

T5 ujmuje zadania jako text-to-text, np. `translate English to German: ...` albo `summarize: ...`. Wspólny format nie znaczy, że cel semantyczny zadań jest ten sam.

In [ ]:
import math,torch
from torch import nn
torch.manual_seed(23);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Attention(nn.Module):
    def __init__(self,d):
        super().__init__();self.q=nn.Linear(d,d);self.k=nn.Linear(d,d);self.v=nn.Linear(d,d);self.o=nn.Linear(d,d)
    def forward(self,qin,kvin,mask=None):
        q,k,v=self.q(qin),self.k(kvin),self.v(kvin);s=q@k.transpose(-2,-1)/math.sqrt(q.size(-1))
        if mask is not None:s=s.masked_fill(~mask,torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
class EncoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.a=Attention(d);self.n1=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n2=nn.LayerNorm(d)
    def forward(self,x):
        x=self.n1(x+self.a(x,x));return self.n2(x+self.ff(x))
class DecoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.sa=Attention(d);self.ca=Attention(d);self.n1=nn.LayerNorm(d);self.n2=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n3=nn.LayerNorm(d)
    def forward(self,y,mem):
        t=y.size(1);mask=torch.tril(torch.ones(t,t,dtype=torch.bool,device=y.device))
        y=self.n1(y+self.sa(y,y,mask));y=self.n2(y+self.ca(y,mem));return self.n3(y+self.ff(y))
enc=torch.randn(2,7,32,device=device);dec=torch.randn(2,4,32,device=device)
print("cross-attention:",Attention(32).to(device)(dec,enc).shape)
print("encoder:",EncoderLayer(32).to(device)(enc).shape)
print("decoder:",DecoderLayer(32).to(device)(dec,enc).shape)

## 2. Cross-attention i teacher forcing
Cross-attention używa zapytań z dekodera ($Q=H_{dec}W_Q$) oraz kluczy i wartości z encodera ($K=H_{enc}W_K$, $V=H_{enc}W_V$). Decoder dostaje cel przesunięty w prawo, by nie widzieć przyszłości. W pełnym treningu potrzebne są maski paddingowe dla źródła i celu.

## 3. Span corruption w T5
Pretraining T5 usuwa fragmenty (spany) wejścia i każe decoderowi odtworzyć je. Sentinel `<extra_id_0>` zastępuje usunięty span, a target zawiera sentinel i treść. To różni się od GPT, które przewiduje następny token całego tekstu. Przykład jest dydaktyczny i nie implementuje próbkowania spanów ani SentencePiece.

In [ ]:
def corrupt(tokens,start,length):
    removed=tokens[start:start+length];sent="<extra_id_0>"
    return tokens[:start]+[sent]+tokens[start+length:],[sent]+removed+["<extra_id_1>"]
s="bright acidity balances ripe citrus fruit".split();inp,target=corrupt(s,2,2)
print("oryginał:",s,"\nwejście encoder:",inp,"\ncel decoder:",target)

## 4. Porównanie rodzin

| Rodzina | Układ | Typowy cel / zastosowanie |
|---|---|---|
| Transformer (2017) | encoder + decoder | tłumaczenie; attention zastępuje rekurencję |
| GPT | decoder-only | next-token i generowanie |
| BERT | encoder-only | reprezentacje dwukierunkowe, pierwotnie masked LM |
| T5 | encoder–decoder | text-to-text, span corruption |
| GPT-3 | decoder-only, duża skala | few-shot/in-context learning opisane w publikacji |
| GPT-4 | szczegóły modelu w dużej mierze nieujawnione | model ogólnego przeznaczenia |
| ChatGPT | asystent dialogowy | system zoptymalizowany do interakcji |

### Zadanie doboru
Uzasadnij wybór dla: (1) klasyfikacji sentymentu, (2) kontynuacji recenzji, (3) tłumaczenia, (4) streszczania, (5) uzupełniania luki. Encoder-only dobrze tworzy reprezentację; decoder-only naturalnie kontynuuje; encoder–decoder oddziela warunki od wyjścia. To typowe dopasowanie, nie twarde ograniczenie.

## Podsumowanie ścieżki
Poznałeś Q/K/V, skalowanie, causal mask, multi-head attention i blok GPT; trenowałeś mini-GPT na recenzjach; omówiłeś skalę, GPT-3/GPT-4/ChatGPT; teraz porównałeś T5 i inne architektury. Dobieraj cel i architekturę do przepływu informacji, formatu zadania, kosztu i danych. Pełne treningi wymagają tokenizera, masek paddingowych, optymalizacji, ewaluacji, checkpointów i kontroli jakości danych.